In [1]:
import os
import sys

PROJECT_PATH = os.path.join(os.getcwd(),'..')
DATA_PATH = os.path.join(PROJECT_PATH,'data')
sys.path.insert(0,PROJECT_PATH)

from dotenv import load_dotenv
from pdf2image import convert_from_path
from src.llms.factory import get_llm
from src.prompts.ingestion_prompts import OCR_MULTIMODAL_PROMPT
from src.settings import Settings, get_settings
from langchain_core.messages import HumanMessage

load_dotenv()

/Users/albertovacas/Desktop/ai_projects/docuagent/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [2]:
import base64

def encode_image(image_path):
    """Convierte una imagen local en una cadena base64."""
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode('utf-8')

In [3]:
pdf_path = DATA_PATH + "/DeepSeek_V4.pdf"
page_num = 39

images = convert_from_path(pdf_path, first_page=page_num, last_page=page_num)

In [4]:
# Guardar temporalmente para que Gemini la procese
temp_image_path = f"{DATA_PATH}/temp_page_{page_num}.png"
images[0].save(temp_image_path, "PNG")
# 2. Convertir esa imagen a Base64
base64_image = encode_image(temp_image_path)

In [26]:
settings = get_settings()
llm = get_llm(model_name=settings.ITT_MODEL_NAME, settings=settings) 

In [ ]:
message = HumanMessage(
    content=[
        {"type": "text", "text": OCR_MULTIMODAL_PROMPT},
        {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{base64_image}"}},
    ]
)

print(f"👁️ Gemini analizando visualmente la página {page_num}...")
response = llm.invoke([message])

👁️ Gemini analizando visualmente la página 39...


In [29]:
print(response.content)

# Table 7 | Comparison among different sizes and modes of DeepSeek-V4 series. "Non-Think", "High", and "Max" denote reasoning effort.

| Benchmark (Metric) | DeepSeek-V4-Flash Non-Think | DeepSeek-V4-Flash High | DeepSeek-V4-Flash Max | DeepSeek-V4-Pro Non-Think | DeepSeek-V4-Pro High | DeepSeek-V4-Pro Max |
| --- | --- | --- | --- | --- | --- | --- |
| MMLU-Pro (EM) | 83.0 | 86.4 | 86.2 | 82.9 | 87.1 | 87.5 |
| SimpleQA-Verified (Pass@1) | 23.1 | 28.9 | 34.1 | 45.0 | 46.2 | 57.9 |
| Chinese-SimpleQA (Pass@1) | 71.5 | 73.2 | 78.9 | 75.8 | 77.7 | 84.4 |
| GPOA Diamond (Pass@1) | 71.2 | 87.4 | 88.1 | 72.9 | 89.1 | 90.1 |
| HLE (Pass@1) | 8.1 | 29.4 | 34.8 | 7.7 | 34.5 | 37.7 |
| LiveCodeBench (Pass@1-COT) | 55.2 | 88.4 | 91.6 | 56.8 | 89.8 | 93.5 |
| Codeforces (Rating) | - | 2816 | 3052 | - | 2919 | 3206 |
| HMMT 2026 Feb (Pass@1) | 40.8 | 91.9 | 94.8 | 31.7 | 94.0 | 95.2 |
| IMOAnswerBench (Pass@1) | 41.9 | 85.1 | 88.4 | 35.3 | 88.0 | 89.8 |
| Apex (Pass@1) | 1.0 | 19.1 | 33.0 | 0.4 | 